# Stress-Testing Helpers on a New Dataset

The helpers built throughout this course were developed and tested on HR analytics data.
This lesson runs the **exact same pipeline** on a different dataset: the **Telco Customer Churn** dataset.

The task is structurally similar — binary classification — but the data is different in ways that matter:

| | HR Analytics | Telco Churn |
|---|---|---|
| Rows | ~5,000 | ~7,000 |
| Positive class rate | ~8% | ~26% |
| Domain | Employee attrition | Customer churn |

The goal is not to build the best model. It is to observe: **do the helpers still behave sensibly on data they have never seen?**


## 1 — Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [ ]:
%pip install -q google-genai pandas scikit-learn matplotlib python-dotenv


In [ ]:
import os
import sys
import urllib.request
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# --- helper paths (same as 6.1) ---
sys.path.append(str(Path("../01-Exploratory-Data-Analysis-Helpers").resolve()))
sys.path.append(str(Path("../02-Preprocessing-Helpers").resolve()))
sys.path.append(str(Path("../03-Modeling-Helpers").resolve()))
sys.path.append(str(Path("../04-Evaluation-Helpers").resolve()))

from eda_summary_helper import eda_summary_helper
from eda_visual_helpers import eda_visual_helper
from preprocessing_pipeline import preprocessing_pipeline
from classification_helper import classification_helper
from classification_eval_helper import (
    compute_classification_metrics,
    flag_anomalies,
)

warnings.filterwarnings("ignore")
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

RAW_PATH = "../../data/telco_churn.csv"
TARGET_COL = "Churn"
GEN_CONFIG = {"temperature": 0.0, "seed": 42}
pd.set_option("display.max_colwidth", None)


## 2 — EDA

### About the Dataset

**Telco Customer Churn** — originally published by IBM as a sample dataset, 
widely used as a benchmark for binary classification.

- **Source:** IBM Sample Datasets (via [treselle-systems/customer_churn_analysis](https://github.com/treselle-systems/customer_churn_analysis/blob/master/WA_Fn-UseC_-Telco-Customer-Churn.csv))
- **Rows:** 7,043 customers
- **Columns:** 21 features + 1 target (`Churn`: Yes / No)
- **Positive class rate:** ~26% churned

**What the data contains:**
- **Demographics** — gender, SeniorCitizen, Partner, Dependents
- **Account info** — tenure, Contract type, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges
- **Services subscribed** — PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies

**Known data quality issue:**  
`TotalCharges` is stored as a string in the raw file — 11 new customers with no charges have a space instead of `0`. 
This must be coerced to numeric before the preprocessing pipeline can run.


In [ ]:
# Download the dataset if not already present
if not Path(RAW_PATH).exists():
    url = "https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"
    Path(RAW_PATH).parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, RAW_PATH)
    print(f"Downloaded to {RAW_PATH}")
else:
    print(f"Found: {RAW_PATH}")


In [ ]:
# TotalCharges is stored as string in the raw file — coerce to numeric.
# Also drop customerID so the preprocessing pipeline doesn't carry an
# identifier column through to the model.
df_raw = pd.read_csv(RAW_PATH)
df_raw["TotalCharges"] = pd.to_numeric(df_raw["TotalCharges"], errors="coerce")
df_raw = df_raw.drop(columns=["customerID"], errors="ignore")
df_raw.to_csv(RAW_PATH, index=False)

print(f"Shape: {df_raw.shape}")
print(
    f"Churn rate: {df_raw[TARGET_COL].value_counts(normalize=True).round(3).to_dict()}"
)
df_raw.head(3)


In [ ]:
eda_result = eda_summary_helper(
    question="Summarise this dataset for a classification task predicting Churn. Flag any data quality issues, class imbalance or columns that look problematic.",
    frame=df_raw,
)
print(eda_result)


In [ ]:
df_numeric = df_raw.select_dtypes(include="number").assign(Churn=df_raw[TARGET_COL])

eda_visual_helper(
    question="Show the distribution of MonthlyCharges.",
    frame=df_numeric,
)

eda_visual_helper(
    question="Show churn counts.",
    frame=df_numeric,
)

eda_visual_helper(
    question="Show the correlation matrix.",
    frame=df_numeric,
)


## 3 — Preprocessing

In [ ]:
result = preprocessing_pipeline(
    raw_path=RAW_PATH,
    target_col=TARGET_COL,
    task="classification",
)
X_train, X_test = result.X_train_enc, result.X_test_enc
y_train, y_test = result.y_train, result.y_test

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(
    f"Class balance (train): {pd.Series(y_train).value_counts(normalize=True).round(3).to_dict()}"
)


## 4 — Modeling

The classification helper selects a model, tunes hyperparameters, and returns predictions.
It explains which model it chose and why it rejected the alternatives.


In [ ]:
results = classification_helper(X_train, X_test, y_train, y_test)

print(f"Model selected : {results['model_name']}")
print(f"Init kwargs    : {results['init_kwargs']}")


## 5 — Evaluation

Metrics are computed from predictions — not generated by the LLM. The LLM then flags anomalies on the metrics dict.


In [ ]:
metrics = compute_classification_metrics(
    y_test,
    results["y_pred"],
    results["y_proba"],
)
pd.DataFrame([metrics])


In [ ]:
# Baseline check — does the model beat always predicting the majority class?
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

pd.DataFrame(
    {
        "metric": ["accuracy", "f1 (weighted)"],
        "dummy": [
            round(dummy.score(X_test, y_test), 3),
            round(f1_score(y_test, dummy.predict(X_test), average="weighted"), 3),
        ],
        "our_model": [
            round(metrics["accuracy"], 3),
            round(f1_score(y_test, results["y_pred"], average="weighted"), 3),
        ],
    }
)


In [ ]:
flags = flag_anomalies(metrics)
pd.DataFrame(flags)
